# Welcome to Part 1 of the NIBLEs Tutorial.

NIBLEs is a python package that enables simulation of MR pulse sequences on arbitrary hardware configurations through numeric solving of the Bloch equations. This tutorial will teach you how to used the tools provided in NIBLEs to perform simulation of MR pulse sequences. It assumes you have a basic knowledge of MR physics, MR pulse sequences, and programming in Python.

To access the NIBLEs code, this file (and all other files in this tutorial) must be located in a directory containing the simCode folder (which contains the source code for the NIBLEs toolset).

## Before beginning this tutorial, please ensure the following software is installed:
* Python version 3.10 or later
* NumPy version 1.24.2 or later
* SciPy version 1.10.1 or later
* MatPlotLib verison 3.0.0 or later

Before we can simulate an MR experiment with NIBLEs, we need to define whatever sample we are studying. For Part 1 of this tutorial, we will demonstrate sample creation in NIBLEs by creating a sample for simple NMR experiments.

To start, we need to import the NIBLEs sample creation tools, found in `./NIBLEs/simCode/samplegen.py`, as well as NumPy. Please do so in the cell below.


In [1]:
# Input your import statements

import numpy as np
import simcode.samplegen as sampgen

Next, we need to define some basic parameters defining our sample:

First we define the dimensions of our imaging volume in meters. Since we'll be starting with some NMR experiments, let's set this sample volume to a 1 cm^3 cube. Please set the metric dimensions (`size_metric`) of our volume to `[0.01, 0.01, 0.01]`.

Now, we will divide our imaging volume into voxels. Again, since we're starting with an NMR sample and will not be using this sample for imaging, let's treat our sample as a single voxel. Please set the voxel dimensions (`size_pixel`) of our volume to `[1, 1, 1]`.

We also need to set the number of magnetization vectors to be used to simulate signal from each voxel. When using NIBLEs in its default configuration, we recommend using a large number of vectors per voxel for NMR experiments. The reasoning for this is laid out in an explainer at the end of Part 2 of this tutorial. For this tutorial, please set `n_vec` = `1024`.

Finally, we need to name our sample and specify where it should be saved. As good practice, we recommend naming a file using the convention `(Name)_(number of vectors)_(Dimensions)` to make it obvious to other users what the sample is and what the properties of the imaging volume are. For this tutorial, we have pre-filled a name for you. We recommend saving samples in a folder named `samples` within the same directory as `simcode`.

In [2]:
# Define the specified parameters
# Sample metric dimensions
size_metric = [0.01, 0.01, 0.01]         

# Sample voxel dimensions
size_pixel = [1, 1, 1] 

# Number of vectors per voxel
n_vec = 1024

# Sample name
sample_name = (f"NIBLEsTutorial_NMR_{n_vec}Vectors_{size_pixel[0]}x{size_pixel[1]}_{size_metric[0]*100}cm")

# Sample storage location
sample_path = "./samples"

With the sample volume defined, we now need to define the properties of the materials present within our sample. This is done using the `Material` class, which stores the relevant information about a single material for sample creation. These properties are:

1) **Proton Density (`pd`)** - The *relative* proton density of each material within a sample. You are free to use any scale or units you wish, but please ensure you are using a consistent system across all materials in a sample. By default, materials dound in the NIBLEs source code normalize pd relative to the material with the higherst pd in the sample.

2) **Chemical Shift (`chem_shift`)** - The chemical shift of the material in question, measured in ppm relative to water.

3) **Irreversable relaxation times (`t1, t2`)** - NMR relaxation times of your material. It is important to note that the relaxation information stored in `Material` can be more complex than a simple static value. NIBLEs simulates relaxation times dynamically as a function of applied field strength, using a user-specified function from `./NIBLEs/simcode/relaxfunc.py`. The `t1` and `t2` variables in `Material` are given to the relaxation function as a set of parameters that define relaxation behaviour. To accomodate for this complexity, `Material` contains 4 different pairs of `t1`/`t2` variables to enable simulation of different relaxation behaviours with the same sample file. We will delve into this in greater detail later in the tutorial, for now we will work with a single set of static relaxation vlaues.

4) **Reversible relaxation time (`t2_star`)** - Reversible $T_2$ effects. As these effects are not directly modeled by the Bloch equations, NIBLEs has an inbuilt system for modelling $T_2^*$ based on a single given value. When creating an instance of `Material`, we only need to define a single static value for $T_2^*$. For more details, again please refer to the NIBLEs paper.


For each material in your sample, you will need to create a separate instance of the Material class. Each instance should be assigned a unique, non-zero integer index that we will use to identify which voxel contains which tissue. For this tutorial, let's create an instance of Material using synthetic values in the code cell below:

* `index` = 1 (A non-zero integer value unique to each material in the sample)
* `pd` = 1 (As our sample will only contain one material, let's set pd to unity)
* `chem_shift` = 0 (Again, only one material so set chemicalShift to zero)
* `t1` = 1 ($T_1$ = 1 second)
* `t2` = 100e-3 ($T_2$ = 100 ms)
* `t2_star` = 10e-3 ($T_2^*$ = 10ms)

Please note that a NIBLEs Material object has additional inputs for field-dependent relaxation calculations that we will work with later in the tutorial. When creating your first material, please simply set the followeing input variables to `0`: `dynamic_t1`, `dynamic_t2`, `t1_alt`, `t2_alt`,`dynamic_t1_alt`, `dynamic_t2_alt`.

In [ ]:
# Create an instance of the 'Material' class
sample_material = sampgen.Material(index = 1,
                               pd = 1,
                               chem_shift = 0,       
                               t1 = 1,
                               t2 = 100e-3,
                               t2_star = 10e-3, 
                               
                               # Alternate T1/T2 variables; not used for
                               # this section of the tutorial
                               dynamic_t1 = 0,
                               dynamic_t2 = 0, 
                               t1_alt = 0, 
                               t2_alt = 0, 
                               dynamic_t1_alt = 0,
                               dynamic_t2_alt = 0)

The final element we need to create a sample is a spatial map of the materials present within the imaging volume. This map should be a 3D numpy array with the same dimensions as the imaging volume `size_pixel`. Each element in the array should be set to the index value `index` assigned to the `Material` filling that voxel of our sample volume. As we only have a single voxel in this example, our spatial map is simply a 3D, one-element array containing the index assigned to `sample_material` above. Please create this in the cell below.

In [4]:
# Create a single element 3D Array
sample_map = np.array([[[1]]])

Now that we have everything we need, we can use the functions in SampleClasses.py to create our sample. In the cell below:
* Start by creating an instance of the `ImagingVolume` class, using the parameters we defined above
* Set the `sample_mask` variable within your `ImagingVolume` class to be the map of the imgaing volume you defined just above.
* Use the built in function `construct_sample` to take the information we've given the solver and construct the data arrays the NIBLEs solver will use to specify individual magnetization vectors and their properties. The only input requiredby this function is a list of each Material in the sample.
* Finally, save the sample to file using the built in function `save_sample`

In [ ]:
# Write your sample creation code here
Volume = sampgen.ImagingVolume(size_metric = size_metric, 
                               size_pixels = size_pixel, n_vectors = n_vec)
Volume.sample_mask = sample_map
Volume.construct_sample(mater_classes = [sample_material])
Volume.save_sample(sample_name, sample_path)

With that, you should now see a new folder in `./NIBLEs/samples` (assuming it didn't already exist) containing a number of files. These files store all the parameters contained in `Material` for each magnetization vector in our sample. Additionally, each of these vectors have been assigned a unique position within the imaging volume, which is also saved to file.

#### Congratulations, you have completed Part 1 of this tutorial. Please continue to Part 2, where we will use the sample we have generated to simulate some basic pulse sequences using NIBLEs!